# Exercise 52 — Compressed files (`gzip`)

## Concept

Large text files (CSV, JSONL, logs) are routinely stored gzipped. The stdlib `gzip` module gives you a drop-in replacement for `open()`:

```python
import gzip

with gzip.open("data.csv.gz", "rt", encoding="utf-8") as f:   # 'rt' = read text
    for line in f:
        ...

with gzip.open("out.csv.gz", "wt", encoding="utf-8") as f:    # 'wt' = write text
    f.write("id,name\n1,Alice\n")
```

Note the mode characters:
- `"rb"` / `"wb"` — binary (default; returns `bytes`)
- `"rt"` / `"wt"` — text (decodes to `str`); pass `encoding="utf-8"`

**Streaming wins big here.** `gzip.open` decompresses on the fly — you never have to decompress the whole file to disk first. A 10GB gzipped CSV reads with constant memory.

### Combining with `csv` / `json`

Both `csv.reader` and the `json` module accept any file-like object — so they happily consume `gzip.open(...)` directly:

```python
import gzip, csv

with gzip.open("data.csv.gz", "rt", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        ...
```

No intermediate decompression step needed.

### Pandas

`pd.read_csv` / `pd.to_csv` auto-detect gzip from the `.gz` extension:

```python
pd.read_csv("data.csv.gz")              # auto-decompressed
df.to_csv("out.csv.gz", index=False)    # auto-gzipped
```

## Setup

Generate a gzipped CSV with ~200 rows so size comparisons are meaningful.

In [ ]:
import gzip
from pathlib import Path

rows = ["id,event,user\n"]
for i in range(200):
    rows.append(f"{i},click,user_{i % 5}\n")

plain = Path("events.csv")
plain.write_text("".join(rows), encoding="utf-8")

with gzip.open("events.csv.gz", "wt", encoding="utf-8") as f:
    f.writelines(rows)

print("plain:    ", plain.stat().st_size, "bytes")
print("gzipped:  ", Path("events.csv.gz").stat().st_size, "bytes")


## Task 1 — Count rows in a gzipped CSV

Write a function that counts data rows (excluding the header) in a `.csv.gz` file. Stream it — do **not** decompress the whole file first.

```python
def count_rows_gz(path: str) -> int:
    ...
```

Expected: `count_rows_gz("events.csv.gz")` → `200`.

In [ ]:
import gzip
import csv

def count_rows_gz(path: str) -> int:
    pass  # TODO

assert count_rows_gz("events.csv.gz") == 200
print("ok")


## Task 2 — Filter and re-compress

Write a function that reads `events.csv.gz`, keeps only the rows where `user == 'user_0'`, and writes them to `events_user0.csv.gz` (still gzipped, with header).

```python
def filter_to_gz(src: str, dst: str, user: str) -> int:
    """Return the number of rows written (excluding header)."""
```

Stream both ends — `gzip.open` for read AND write, `csv.DictReader` + `csv.DictWriter`. Constant memory.

In [ ]:
import gzip
import csv

def filter_to_gz(src: str, dst: str, user: str) -> int:
    pass  # TODO

n = filter_to_gz("events.csv.gz", "events_user0.csv.gz", "user_0")
assert n == 40   # 200 rows, 5 distinct users → 40 per user
assert count_rows_gz("events_user0.csv.gz") == 40
print("ok")


## Task 3 — Pandas auto-detection

Write a function that uses pandas to load `events.csv.gz` directly and returns the row count.

```python
def count_rows_pandas(path: str) -> int:
    ...
```

One line of pandas — no manual gzip handling. Verify it agrees with `count_rows_gz`.

Note: pandas auto-detects gzip from the `.gz` extension. For a non-standard extension you'd pass `compression="gzip"` explicitly.

In [ ]:
import pandas as pd

def count_rows_pandas(path: str) -> int:
    pass  # TODO

assert count_rows_pandas("events.csv.gz") == 200
print("ok")
